In [0]:
import requests
import hashlib
import json
from datetime import datetime, timezone
from pyspark.sql import Row

##  MOCKAROO DATA READING

In [0]:
MOCKAROO_SCHEMA_ID = "7640ba60"
MOCKAROO_API_KEY = dbutils.secrets.get(scope="mia-secrets", key="mockaroo-api-key")
MOCKAROO_ROW_COUNT = 1000
BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

In [0]:
url = f"https://api.mockaroo.com/api/{MOCKAROO_SCHEMA_ID}"
params = {"count": MOCKAROO_ROW_COUNT, "key": MOCKAROO_API_KEY}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
mockaroo_raw = response.json()

print(f"Status: {response.status_code}")
print(f"Records pulled: {len(mockaroo_raw)}")
print(f"Sample record: {mockaroo_raw[0]}")

Status: 200
Records pulled: 1000
Sample record: {'product_id ': 1, 'product_name ': 'Robot Vacuum Cleaner', 'category ': 'Grocery', 'price ': 299.99, 'brand': 'Roombo', 'stock_quantity ': 317, 'sku ': 'd728a0c0-93eb-4339-96a7-ab472b25d9e3', 'created_at ': '12/14/2025'}


## DUMMY JSON DATA READING

In [0]:
DUMMYJSON_BASE = "https://dummyjson.com"
DUMMYJSON_PAGE_LIMIT = 100

In [0]:
def pull_all_dummyjson(endpoint: str, data_key: str) -> list:
    all_records = []
    skip = 0
    while True:
        url = f"{DUMMYJSON_BASE}/{endpoint}"
        params = {"limit": DUMMYJSON_PAGE_LIMIT, "skip": skip}
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        payload = resp.json()

        batch = payload.get(data_key, [])
        all_records.extend(batch)

        total = payload.get("total", len(all_records))
        skip += DUMMYJSON_PAGE_LIMIT
        if skip >= total:
            break
    return all_records

In [0]:
dj_products_raw = pull_all_dummyjson("products", "products")
print(f"Products pulled: {len(dj_products_raw)}")
print(f"Sample: {dj_products_raw[0]['title']}, {dj_products_raw[0]['category']}, {dj_products_raw[0]['price']}")

Products pulled: 194
Sample: Essence Mascara Lash Princess, beauty, 9.99


In [0]:
dj_users_raw = pull_all_dummyjson("users", "users")
print(f"Users pulled: {len(dj_users_raw)}")
print(f"Sample: {dj_users_raw[0]['firstName']} {dj_users_raw[0]['lastName']}, {dj_users_raw[0]['email']}, {dj_users_raw[0]['address']['city']}")

Users pulled: 208
Sample: Emily Johnson, emily.johnson@x.dummyjson.com, Phoenix


In [0]:
dj_carts_raw = pull_all_dummyjson("carts", "carts")
print(f"Carts pulled: {len(dj_carts_raw)}")
print(f"Sample: userId={dj_carts_raw[0]['userId']}, totalProducts={dj_carts_raw[0]['totalProducts']}, total={dj_carts_raw[0]['total']}")

Carts pulled: 208
Sample: userId=1, totalProducts=4, total=13037.88


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS mia_catalog;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS mia_catalog.bronze;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS mia_catalog.bronze.mia_landing;

In [0]:
display(spark.sql("SHOW VOLUMES IN mia_catalog.bronze"))

database,volume_name
bronze,mia_landing


### MOCKAROO READSTREAM AND WRTIESTREAM

In [0]:
landing_path = "/Volumes/mia_catalog/bronze/mia_landing/mockaroo_products"
checkpoint_path = "/Volumes/mia_catalog/bronze/mia_landing/_checkpoints/mockaroo_products"
dbutils.fs.mkdirs(landing_path)

file_path = f"{landing_path}/mockaroo_{BATCH_ID}.json"
dbutils.fs.put(file_path, "\n".join(json.dumps(r) for r in mockaroo_raw), overwrite=True)

print(f"Landed file: {file_path}")

Wrote 221396 bytes.
Landed file: /Volumes/mia_catalog/bronze/mia_landing/mockaroo_products/mockaroo_20260811093520.json


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col

# Clean slate
spark.sql("DROP TABLE IF EXISTS bronze_mockaroo_products_stream")
dbutils.fs.rm(checkpoint_path, recurse=True)

# Schema matching the ACTUAL raw JSON keys (with trailing spaces)
mockaroo_schema = StructType([
    StructField("product_id ", IntegerType(), True),
    StructField("product_name ", StringType(), True),
    StructField("category ", StringType(), True),
    StructField("price ", DoubleType(), True),
    StructField("brand", StringType(), True),
    StructField("stock_quantity ", IntegerType(), True),
    StructField("sku ", StringType(), True),
    StructField("created_at ", StringType(), True),
])

# Read raw, then immediately rename via explicit select — no ambiguity, no stale reference
raw_stream = (
    spark.readStream
    .format("json")
    .schema(mockaroo_schema)
    .load(landing_path)
)

mockaroo_stream_df = raw_stream.select(
    col("product_id ").alias("product_id"),
    col("product_name ").alias("product_name"),
    col("category ").alias("category"),
    col("price ").alias("price"),
    col("brand").alias("brand"),
    col("stock_quantity ").alias("stock_quantity"),
    col("sku ").alias("sku"),
    col("created_at ").alias("created_at"),
)

query = (
    mockaroo_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("bronze_mockaroo_products_stream")
)

query.awaitTermination()
print("Stream completed")

Stream completed


In [0]:
%sql
DESCRIBE DETAIL bronze_mockaroo_products_stream;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,73b47fb4-e6a8-4eb3-b07b-e8be93901782,workspace.default.bronze_mockaroo_products_stream,null,,2026-08-11T09:54:13.867Z,2026-08-11T09:54:19.000Z,List(),List(),1,40500,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-bbca348f-a44d-4d5d-96ee-0d46ae4005ae, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-6ef33bc6-daea-4043-89fd-f6f5460e21d5)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
display(spark.sql("SELECT * FROM bronze_mockaroo_products_stream LIMIT 5"))

product_id,product_name,category,price,brand,stock_quantity,sku,created_at
1,Non-Stick Grill Pan,Tools,39.99,Demizz,321,1b7f8dbf-1fdf-41ff-96bd-d57bd3033331,12/24/2025
2,Outdoor Portable Fire Pit,Toys,149.99,Rhyloo,242,e9c54758-85a8-451c-be15-88fc74fc803c,3/2/2026
3,Lemon Garlic Shrimp,Beauty,8.99,Dynabox,169,7f84204e-0575-4875-a2df-ceaa25e3242f,11/1/2025
4,Savory Trail Mix,Music,4.29,Katz,398,15afe106-efcc-4978-b75b-7917a53503a9,3/26/2026
5,Digital Bullet Journal,Books,24.99,Divanoodle,488,cb5e5542-1637-48c1-a0aa-dd62c8646c88,5/7/2026


### DUMMYJSON READSTREAM AND WRITESTREAM

In [0]:
def land_dummyjson_entity(records: list, entity_name: str, id_field: str = "id"):
    """Wraps each raw record in a flat metadata envelope and lands it as a JSON file."""
    entity_landing_path = f"/Volumes/mia_catalog/bronze/mia_landing/dummyjson_{entity_name}"
    dbutils.fs.mkdirs(entity_landing_path)

    envelopes = []
    for r in records:
        natural_key = str(r.get(id_field, ""))
        envelope = {
            "record_id": natural_key,
            "raw_payload": json.dumps(r),
            "record_hash": hashlib.md5(json.dumps(r, sort_keys=True).encode("utf-8")).hexdigest(),
            "batch_id": BATCH_ID,
            "source_system": "dummyjson",
            "source_entity": entity_name,
            "load_ts": datetime.now(timezone.utc).isoformat(),
        }
        envelopes.append(envelope)

    file_path = f"{entity_landing_path}/{entity_name}_{BATCH_ID}.json"
    dbutils.fs.put(file_path, "\n".join(json.dumps(e) for e in envelopes), overwrite=True)
    print(f"Landed {len(envelopes)} {entity_name} records to {file_path}")
    return entity_landing_path

In [0]:
dj_products_landing_path = land_dummyjson_entity(dj_products_raw, "products", id_field="id")
dj_users_landing_path = land_dummyjson_entity(dj_users_raw, "users", id_field="id")
dj_carts_landing_path = land_dummyjson_entity(dj_carts_raw, "carts", id_field="id")

Wrote 395176 bytes.
Landed 194 products records to /Volumes/mia_catalog/bronze/mia_landing/dummyjson_products/products_20260811093520.json
Wrote 401729 bytes.
Landed 208 users records to /Volumes/mia_catalog/bronze/mia_landing/dummyjson_users/users_20260811093520.json
Wrote 290835 bytes.
Landed 208 carts records to /Volumes/mia_catalog/bronze/mia_landing/dummyjson_carts/carts_20260811093520.json


In [0]:
envelope_schema = StructType([
    StructField("record_id", StringType(), True),
    StructField("raw_payload", StringType(), True),
    StructField("record_hash", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("source_entity", StringType(), True),
    StructField("load_ts", StringType(), True),
])

In [0]:
def stream_dummyjson_entity_to_bronze(entity_name: str, landing_path_for_entity: str):
    checkpoint = f"/Volumes/mia_catalog/bronze/mia_landing/_checkpoints/dummyjson_{entity_name}"
    target_table = f"bronze_dummyjson_{entity_name}"

    stream_df = (
        spark.readStream
        .format("json")
        .schema(envelope_schema)
        .load(landing_path_for_entity)
    )

    query = (
        stream_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(target_table)
    )

    query.awaitTermination()
    print(f"[{entity_name}] Stream completed -> {target_table}")

In [0]:
stream_dummyjson_entity_to_bronze("products", dj_products_landing_path)
stream_dummyjson_entity_to_bronze("users", dj_users_landing_path)
stream_dummyjson_entity_to_bronze("carts", dj_carts_landing_path)

[products] Stream completed -> bronze_dummyjson_products
[users] Stream completed -> bronze_dummyjson_users
[carts] Stream completed -> bronze_dummyjson_carts


In [0]:
for entity in ["products", "users", "carts"]:
    print(f"--- {entity} ---")
    display(spark.sql(f"SELECT record_id, source_entity, load_ts, substring(raw_payload, 1, 80) as payload_preview FROM bronze_dummyjson_{entity} LIMIT 3"))

--- products ---


record_id,source_entity,load_ts,payload_preview
1,products,2026-08-11T09:57:01.387524+00:00,"{""id"": 1, ""title"": ""Essence Mascara Lash Princess"", ""description"": ""The Essence"
2,products,2026-08-11T09:57:01.387594+00:00,"{""id"": 2, ""title"": ""Eyeshadow Palette with Mirror"", ""description"": ""The Eyeshado"
3,products,2026-08-11T09:57:01.387699+00:00,"{""id"": 3, ""title"": ""Powder Canister"", ""description"": ""The Powder Canister is a f"


--- users ---


record_id,source_entity,load_ts,payload_preview
1,users,2026-08-11T09:57:02.183187+00:00,"{""id"": 1, ""firstName"": ""Emily"", ""lastName"": ""Johnson"", ""maidenName"": ""Smith"", ""a"
2,users,2026-08-11T09:57:02.183258+00:00,"{""id"": 2, ""firstName"": ""Michael"", ""lastName"": ""Williams"", ""maidenName"": """", ""age"
3,users,2026-08-11T09:57:02.183317+00:00,"{""id"": 3, ""firstName"": ""Sophia"", ""lastName"": ""Brown"", ""maidenName"": """", ""age"": 4"


--- carts ---


record_id,source_entity,load_ts,payload_preview
1,carts,2026-08-11T09:57:02.932440+00:00,"{""id"": 1, ""products"": [{""id"": 162, ""title"": ""Blue Frock"", ""price"": 29.99, ""quant"
2,carts,2026-08-11T09:57:02.932486+00:00,"{""id"": 2, ""products"": [{""id"": 86, ""title"": ""Man Short Sleeve Shirt"", ""price"": 19"
3,carts,2026-08-11T09:57:02.932549+00:00,"{""id"": 3, ""products"": [{""id"": 24, ""title"": ""Fish Steak"", ""price"": 14.99, ""quanti"


In [0]:
tables = [
    "bronze_mockaroo_products_stream",
    "bronze_dummyjson_products",
    "bronze_dummyjson_users",
    "bronze_dummyjson_carts",
]

for t in tables:
    count = spark.sql(f"SELECT count(*) as cnt FROM {t}").collect()[0]["cnt"]
    print(f"{t}: {count} rows")

bronze_mockaroo_products_stream: 1000 rows
bronze_dummyjson_products: 194 rows
bronze_dummyjson_users: 208 rows
bronze_dummyjson_carts: 208 rows


In [0]:
display(spark.sql("DESCRIBE bronze_dummyjson_products"))

col_name,data_type,comment
record_id,string,null
raw_payload,string,null
record_hash,string,null
batch_id,string,null
source_system,string,null
source_entity,string,null
load_ts,string,null
